In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import os.path as op
import yaml
import pandas as pd

In [ ]:


def read_all_results_yaml(root_dir, select = ('netmap_config.yaml', 'netmap_config_1.yaml')):
    """
    Reads all YAML files in a directory tree starting from root_dir, processes
    the nested structure (including multiple top-level keys), and concatenates
    the results into a single DataFrame.

    Args:
        root_dir (str): The path to the root directory.
        select (tuple or str): File extension or full filename to select.
                               Defaults to common YAML configuration names.

    Returns:
        pandas.DataFrame: A DataFrame containing the processed data from all YAML files.
                          Returns an empty DataFrame if no files are found or an error occurs.
    """
    all_results_dfs = []

    # Normalize the root_dir path
    root_dir = os.path.abspath(root_dir)

    # Check if the root directory exists
    if not os.path.isdir(root_dir):
        print(f"Error: Directory '{root_dir}' not found.")
        return pd.DataFrame() # Return an empty DataFrame

    if isinstance(select, str):
        select = (select,)

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            # Check if the filename ends with any of the selected strings/extensions
            if any(filename.endswith(s) for s in select):
                filepath = op.join(dirpath, filename)
                print(f"Processing: {filepath}")

                try:
                    with open(filepath, 'r') as f:
                        yaml_content = yaml.safe_load(f)

                    if not yaml_content or not isinstance(yaml_content, dict):
                        continue # Skip empty or invalid YAML file

                    # The directory one level up is 'CONFIG'
                    config_dir = op.basename(op.dirname(dirpath))
                    
                    # Iterate through all top-level keys (e.g., netmap_config_1, netmap_config_2, etc.)
                    for config_key, nested_data in yaml_content.items():
                        
                        if not isinstance(nested_data, dict):
                            print(f"Skipping key '{config_key}' in {filename}: not a dictionary.")
                            continue

                        records = []
                        
                        # Extract common metadata for this configuration key
                        clustering_score = nested_data.get('clustering_score')
                        total_number_edges = nested_data.get('total_number_edges')

                        # Iterate through the network-specific keys (like net_53_1113)
                        for net_key, net_data in nested_data.items():
                            
                            # Skip known non-network metadata fields
                            if net_key in ['clustering_score', 'total_number_edges']:
                                continue
                            
                            if not isinstance(net_data, dict):
                                print(f"Skipping network key '{net_key}' under '{config_key}': not a dictionary.")
                                continue

                            # Create a record for the DataFrame
                            record = {
                                'net_name': net_key,           # e.g., net_53_1113
                                'config_key': config_key,      # e.g., netmap_config_1
                                'filename': filename,
                                # Add the common metadata
                                'clustering_score': clustering_score,
                                'total_number_edges': total_number_edges,
                            }
                            # Add the network-specific data (edge_overlap, net_size, etc.)
                            record.update(net_data)
                            
                            records.append(record)

                        # Create a DataFrame from the extracted records for this config_key
                        if records:
                            net_dir = op.basename(dirpath)  
                            # The directory one level up is 'CONFIG'
                            config_dir = op.basename(op.dirname(dirpath))
                        


                            overlaps_df = pd.DataFrame(records)
                            # Add the directory name (similar to your original 'net' column)
                            overlaps_df['dataset'] = op.basename(op.dirname(filepath))
                            overlaps_df['config_dir'] = config_dir
                            all_results_dfs.append(overlaps_df)


                except Exception as e:
                    print(f"Error reading or processing {filepath}: {e}")
                    continue

    if all_results_dfs:
        # Concatenate all DataFrames into one
        all_results = pd.concat(all_results_dfs, ignore_index=True)
    else:
        all_results = pd.DataFrame()

    all_results['n_clusters'] = all_results['config_dir'].apply(recode_number_of_datasets)


    return all_results

def recode_number_of_datasets(config_item):
    if config_item == 'config_easy':
        return 2
    if config_item == 'config_noise':
        return 2
    if config_item == 'config_three':
        return 3
    if config_item == 'config_three_noise':
        return 3
    if config_item == 'config_five':
        return 5
    if config_item == 'config_ten':
        return 10


In [21]:
df_results_leiden = read_all_results_yaml('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/', select=('clustering_score_leiden.json'))
df_results_leiden.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/clustering_scores_leiden.tsv', sep = '\t', index=False)

df_results = read_all_results_yaml('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/', select=('clustering_score.json'))
df_results.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/clustering_score.tsv', sep = '\t', index = False)

df_results_tf = read_all_results_yaml('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/', select=('clustering_score_tf.json'))
df_results_tf.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/clustering_score_tf.tsv', sep = '\t', index = False)



Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/config_easy/net_53_11196_net_70_11431_net_84_9903/clustering_score_leiden.json
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/config_easy/net_59_10488_net_115_11153_net_84_11226/clustering_score_leiden.json
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/config_easy/net_172_10626_net_89_11634_net_76_10367/clustering_score_leiden.json
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/config_easy/net_60_10082_net_64_11307_net_84_11226/clustering_score_leiden.json
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/config_easy/net_133_10773_net_82_10152_net_72_10551/clustering_score_leiden.json
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/config_easy/net_98_11932_net_51_10906_net_60_10082/clustering_score_leiden.json
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/

/tmp/ipykernel_2830660/2506730165.py:105: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_results = pd.concat(all_results_dfs, ignore_index=True)


Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/config_easy/net_70_11431_net_60_10082_net_135_11054/clustering_score.json
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/config_easy/net_53_11196_net_70_11431_net_84_9903/clustering_score.json
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/config_easy/net_59_10488_net_115_11153_net_84_11226/clustering_score.json
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/config_easy/net_172_10626_net_89_11634_net_76_10367/clustering_score.json
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/config_easy/net_60_10082_net_64_11307_net_84_11226/clustering_score.json
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/config_easy/net_133_10773_net_82_10152_net_72_10551/clustering_score.json
Processing: /data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/config_easy/net_98_11932_net_51_10906_net